# Mini Workshop — ONNX

> *PyTorch is the language you wrote your model in. ONNX is the language your model travels in.*

In this Mini, we'll:
1. **Run** SceneSeg (Autoware perception) **as PyTorch** and time it
2. **Export** it to ONNX with `torch.onnx.export()`
3. **Run** the ONNX with ONNX Runtime, verify the math matches, and time it again

The big Workshop applies the same export, then takes the ONNX further into TensorRT FP16 and INT8.


## Setup


In [ ]:
!pip install onnxruntime-gpu gdown onnxscript onnx -q


In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T
import numpy as np
import cv2
import time
import warnings
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import onnxruntime as ort
import onnx
import glob, os

from SceneSeg import SceneSegNetwork

# Silence the 'legacy TorchScript-based ONNX' deprecation warning — we use
# dynamo=False on purpose for plain CNNs (see Bonus_ONNX_Dynamo for details).
warnings.filterwarnings("ignore", message=".*legacy TorchScript-based ONNX.*")

print(f"PyTorch {torch.__version__} | ORT {ort.__version__} | ONNX {onnx.__version__}")


## Download — SceneSeg weights & Waymo frames

We grab the **`.pth` state dict** (regular PyTorch weights). Combined with the model class in `SceneSeg.py` (already in the repo), we get a normal `nn.Module` — exactly what the ONNX exporter wants.


In [ ]:
!mkdir -p /content/data /content/models

!gdown '1vCZMdtd8ZbSyHn1LCZrbNKMK7PQvJHxj' -O /content/models/SceneSeg.pth

!wget -qq https://optical-flow-data.s3.eu-west-3.amazonaws.com/waymo_images.zip -O /content/data/waymo.zip
!unzip -qq /content/data/waymo.zip -d /content/data/

print(f"\nWeights: {os.path.getsize('/content/models/SceneSeg.pth')/1e6:.1f} MB")


In [ ]:
# SceneSeg expects 320 x 640 input.
H, W = 320, 640


## Load a Waymo frame


In [ ]:
frames = sorted(glob.glob('/content/data/night/front_images_night/*.jpg'))
frame_path = frames[100]

frame_bgr = cv2.imread(frame_path)
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

preprocess = T.Compose([
    T.Resize((H, W)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
x    = preprocess(Image.fromarray(frame_rgb)).unsqueeze(0)   # (1,3,H,W) normalized
x_np = x.numpy()                                             # numpy for ONNX Runtime

plt.imshow(cv2.resize(frame_rgb, (W, H))); plt.axis('off')
plt.title('Input: Waymo driving frame'); plt.show()


In [ ]:
COLORS = np.array([
    [240,  40,  40],   # 0 background
    [180,  60, 200],   # 1 foreground (cars / pedestrians)
    [ 80, 200,  80],   # 2 drivable road
], dtype=np.uint8)


from tqdm.auto import tqdm

def bench(fn, label, n=10):
    """Warmup + n timed runs. Shows a progress bar so it doesn't look frozen."""
    fn()
    times = []
    for _ in tqdm(range(n), desc=label, leave=False):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    times = np.array(times)
    print(f"{label:20}  {times.mean():6.1f} ms  ({1000/times.mean():5.1f} FPS)")
    return times.mean()


---
## Part 1 — Run as PyTorch

Load the network class, load the trained weights into it, run a frame through, and **time it**. The latency number is the baseline we'll compare against in Part 3.


In [ ]:
model = SceneSegNetwork()
state_dict = torch.load('/content/models/SceneSeg.pth', map_location='cpu', weights_only=True)
model.load_state_dict(state_dict)
model.eval()

with torch.no_grad():
    pt_logits = model(x).numpy()
pt_map = np.argmax(pt_logits[0], axis=0).astype(np.uint8)

print(f"Output shape: {pt_logits.shape}\n")

# Visual
input_img = cv2.resize(frame_rgb, (W, H))
overlay   = cv2.addWeighted(input_img, 0.5, COLORS[pt_map], 0.5, 0)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].imshow(input_img);     axes[0].set_title('Input frame');           axes[0].axis('off')
axes[1].imshow(COLORS[pt_map]); axes[1].set_title('PyTorch — segmentation'); axes[1].axis('off')
axes[2].imshow(overlay);       axes[2].set_title('PyTorch — overlay');     axes[2].axis('off')
plt.tight_layout(); plt.show()

# Benchmark
def run_pt():
    with torch.no_grad():
        return model(x)
pt_ms = bench(run_pt, 'PyTorch (CPU)')


---
## Part 2 — Export to ONNX

`torch.onnx.export()` needs four things:
1. **The model** (in `eval()` mode)
2. **A dummy input** — same shape the model expects, used to trace the graph
3. **An output filename**
4. **An opset version** — opset 17 is a safe modern choice for TensorRT

We pass `dynamo=False` to use the legacy exporter. SceneSeg is a plain CNN — both exporters produce equivalent ONNX. For a model where `dynamo=True` actually wins (transformers, attention), see `Bonus_ONNX_Dynamo.ipynb`.


In [ ]:
# TODO 1 — Build a dummy input matching the model's expected shape (1, 3, H, W).
dummy = ...

# TODO 2 — Call torch.onnx.export with:
#   - model and dummy
#   - output path '/content/models/SceneSeg.onnx'
#   - opset_version=18  (matches what the modern exporter natively produces)
#   - input_names=['image'], output_names=['segmentation']
#   - dynamo=False
torch.onnx.export(...)

# TODO 3 — Validate the file with onnx.checker.check_model and print its size.
# Sanity: SceneSeg should be ~30 MB. If much smaller, list /content/models/
# to see if weights ended up in a separate .data file.


---
## Part 3 — Run YOUR ONNX

Same input, same expected output — through ONNX Runtime instead of PyTorch. We compare visually, numerically, and on speed.


In [ ]:
# TODO 4 — Build an InferenceSession on '/content/models/SceneSeg.onnx'.
# Bonus: pass a SessionOptions() with graph_optimization_level = ORT_ENABLE_ALL.
sess = ...

# TODO 5 — Run inference on x_np (input name: 'image'). Take argmax to get a seg map.
ort_logits = ...
ort_map    = ...

# TODO 6 — Visualize: input | PyTorch overlay | ONNX overlay (3 panels).

# TODO 7 — Pixel agreement (pt_map == ort_map).mean() * 100 — print and verdict.

# TODO 8 — Benchmark with the bench() helper. Compare to pt_ms from Part 1.


---
## Bonus — What's inside the file?

ONNX files are graphs: nodes (operators) connected by tensors. Quick look:


In [ ]:
m = onnx.load('/content/models/SceneSeg.onnx')

op_counts = {}
for node in m.graph.node:
    op_counts[node.op_type] = op_counts.get(node.op_type, 0) + 1

total_ops  = sum(op_counts.values())
unique_ops = len(op_counts)
top_5      = sorted(op_counts.items(), key=lambda kv: -kv[1])[:5]

print(f"Total nodes:       {total_ops}")
print(f"Unique op types:   {unique_ops}")
print(f"Inputs / outputs:  {len(m.graph.input)} / {len(m.graph.output)}")
print(f"\nTop 5 operators:")
for op, n in top_5:
    print(f"  {op:<20} {n}")

print(f"\nTip: open SceneSeg.onnx in https://netron.app for the visual graph.")


---
## 🎯 Your Turn — Export your Module 1 models to ONNX

In earlier modules, you built three optimized DeepLab variants:
- A **pruned** model (Pruning module)
- A **statically quantized** model (Quantization module)
- A **distilled** student model (Knowledge Distillation module)

All three are real wins — **in PyTorch**. To ship any of them, they need to leave PyTorch. That means ONNX, using exactly the pattern you just learned.

### The challenge

Pick **at least one** of the three (the distilled student is the easiest place to start) and:

1. **Export** it to ONNX with `torch.onnx.export(...)` — same call as Part 2.
2. **Validate** with `onnx.checker.check_model`.
3. **Run** it with ONNX Runtime and compare pixel agreement vs the PyTorch original. Aim for >99%.
4. **Benchmark** ONNX vs PyTorch with the same `bench()` helper.

### Stretch — export all three

Compare the file sizes and CPU latencies of the resulting `.onnx` files. Which optimization produced the smallest deployable artifact? The fastest? Are they the same model? Explain.

### Hints

- **Pruned models** export trivially — pruning sets weights to zero, the graph is unchanged.
- **Distilled students** export trivially — they're smaller `nn.Module`s, nothing special.
- **Statically quantized models** are trickier. PyTorch's `torch.quantization` produces models with `QuantStub`/`DeQuantStub` and `quint8` tensors. Exporting needs `opset_version >= 13` and yields the QDQ format. If you hit walls, that's expected — the Workshop quantizes at deployment time with TensorRT for exactly this reason.


---
## What's next

You exported a PyTorch model to ONNX, ran it with ONNX Runtime, verified the math survived, and measured the speed difference on CPU.

In the Workshop (`Model_Deployment.ipynb`), we take this same ONNX further:
1. Compile to a **TensorRT engine** for the GPU (this is where the speedups get serious)
2. Calibrate to **INT8** with real driving frames
3. Benchmark all 5 runtimes side by side, render a comparison video

You'll also see the **production wrapper pattern** there — bake normalization and argmax into the ONNX so deployment code never needs to know what mean/std the model trained with. We skipped it here to keep the Mini focused on the export itself.

See you in the Workshop.
